In [2]:
pip install ucimlrepo

In [9]:
import numpy as np
from ucimlrepo import fetch_ucirepo
import pandas as pd
from tabulate import tabulate
import joblib

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

def print_metrics(ml_name,y_test, y_pred, y_prob):
  # 5. Compute evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  auc_score = roc_auc_score(y_test, y_prob)
  precision = precision_score(y_test, y_pred)
  recall = recall_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  mcc = matthews_corrcoef(y_test, y_pred)
  #Store results in list
  my_list=[ml_name,accuracy,auc_score,precision,recall,f1,mcc]

  # 6. Print the results
  print(f"Accuracy: {accuracy:.4f}")
  print(f"AUC Score: {auc_score:.4f}")
  print(f"Precision: {precision:.4f}")
  print(f"Recall: {recall:.4f}")
  print(f"F1 Score: {f1:.4f}")
  print(f"Matthews Correlation Coefficient: {mcc:.4f}")
  return my_list

# fetch dataset
cdc_diabetes_health_indicators = fetch_ucirepo(id=891)

# data (as pandas dataframes) and slice to 10,000 samples
X = cdc_diabetes_health_indicators.data.features.iloc[:10000]
y = cdc_diabetes_health_indicators.data.targets.iloc[:10000]

y = y.values.ravel()

# 2. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Feature Scaling (Crucial for KNN, Logistic Regression, and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Initialize and train the Logistic Regression model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# 4. Generate predictions
y_pred = model.predict(X_test_scaled)          # Class labels (0 or 1) for most metrics
y_prob = model.predict_proba(X_test_scaled)[:, 1]  # Probabilities for the positive class (needed for AUC)
print("Predictions for Logistic Regression Model")
op_list=print_metrics("Logistic Regression",y_test, y_pred, y_prob)
final_list=[op_list]

# 3. Initialize and train the Decision Tree Classifier
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train_scaled, y_train)

# 4. Generate predictions
y_pred_dt = clf.predict(X_test_scaled)
y_pred_proba_dt = clf.predict_proba(X_test_scaled)[:, 1]

print("\nPredictions for Decision Tree Classifier")
op_list=print_metrics("Decision Tree",y_test, y_pred_dt, y_pred_proba_dt)
final_list.append(op_list)
# 4. Initialize and train the K-Nearest Neighbors classifier
k = 5
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train_scaled, y_train)

# 5. Make predictions
y_pred_knn = knn.predict(X_test_scaled)
y_pred_proba_knn = knn.predict_proba(X_test_scaled)[:, 1]

print("\nPredictions for K nearest neighbour Classifier")
op_list=print_metrics("KNN",y_test, y_pred_knn, y_pred_proba_knn)
final_list.append(op_list)

# 2. Train the Gaussian Naive Bayes model
nb_classifier = GaussianNB()
nb_classifier.fit(X_train_scaled, y_train)

# 3. Make predictions and get probability estimates
y_pred_nb = nb_classifier.predict(X_test_scaled)
y_proba_nb = nb_classifier.predict_proba(X_test_scaled)[:, 1]

print("\nPredictions for Naive Bayes Gaussian Classifier")
op_list=print_metrics("Naive Bayes",y_test, y_pred_nb, y_proba_nb)
final_list.append(op_list)

# 4. Instantiate and train the SVM Classifier
svm_classifier = SVC(kernel='rbf', probability=True, random_state=42)
svm_classifier.fit(X_train_scaled, y_train)

# 5. Predict hard labels and probability scores on the test set
y_pred_svm = svm_classifier.predict(X_test_scaled)
y_prob_svm = svm_classifier.predict_proba(X_test_scaled)[:, 1]

print("\nPredictions for SVM Classifier")
op_list=print_metrics("Support Vector Machine",y_test, y_pred_svm, y_prob_svm)
final_list.append(op_list)
headers=["ML Algorithm","Accuracy","AUC Score","Precision","Recall","F1 Score","MCC"]

#df=pd.DataFrame(final_list,columns=headers)
#print(df)

# Print table with a specific format (e.g., "grid", "fancy_grid", "markdown")
print(tabulate(final_list, headers=headers, tablefmt="fancy_grid"))

joblib.dump(model, 'lr.pkl')
joblib.dump(clf, 'dt.pkl')
joblib.dump(knn, 'knn.pkl')
joblib.dump(nb_classifier, 'nb_classifier.pkl')
joblib.dump(svm_classifier, 'svm_classifier.pkl')

# Convert the scaled NumPy array back to a pandas DataFrame before exporting
test_df = pd.DataFrame(X_test_scaled, columns=X.columns)
test_df["Diabetes"] = y_test

# Export to CSV
test_df.to_csv("test_data1.csv", index=False)

Predictions for Logistic Regression Model
Accuracy: 0.8357
AUC Score: 0.8036
Precision: 0.4727
Recall: 0.1612
F1 Score: 0.2404
Matthews Correlation Coefficient: 0.2042

Predictions for Decision Tree Classifier
Accuracy: 0.7630
AUC Score: 0.6001
Precision: 0.3026
Recall: 0.3595
F1 Score: 0.3286
Matthews Correlation Coefficient: 0.1870

Predictions for K nearest neighbour Classifier
Accuracy: 0.8307
AUC Score: 0.7031
Precision: 0.4500
Recall: 0.2231
F1 Score: 0.2983
Matthews Correlation Coefficient: 0.2314

Predictions for Naive Bayes Gaussian Classifier
Accuracy: 0.7507
AUC Score: 0.7686
Precision: 0.3358
Recall: 0.5579
F1 Score: 0.4193
Matthews Correlation Coefficient: 0.2870

Predictions for SVM Classifier
Accuracy: 0.8403
AUC Score: 0.7122
Precision: 0.5281
Recall: 0.0971
F1 Score: 0.1640
Matthews Correlation Coefficient: 0.1743
╒════════════════════════╤════════════╤═════════════╤═════════════╤═══════════╤════════════╤══════════╕
│ ML Algorithm           │   Accuracy │   AUC Score │